# Economic Indicators Module Test

## Testing Simplified Employment and Income Modeling

This notebook validates the economic indicators module with:
- 🏢 **Employment Tracking**: Labor force participation and employment rates
- 💰 **Income Modeling**: Average income levels by demographics
- 🧮 **Calculator Components**: Employment rates and total income calculations
- 🔗 **Population Dependencies**: Integration with population dynamics module
- 📊 **Economic Visualizations**: Employment trends and income analysis

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("✅ Libraries imported successfully!")

In [ ]:
# Add the sd_toolkit to the path
import sys
import os
sys.path.insert(0, os.path.join('..', '..', '..', 'sd_toolkit'))

# Import System Dynamics Toolkit components
from sd_toolkit.config import YAMLSystemBuilder, TemplateManager, rate_registry
from sd_toolkit.engine.system import SystemModel
from sd_toolkit.engine.elements import Stock, Flow, Auxiliary, Calculator
from sd_toolkit.data.loader import SpatioTemporalData
from sd_toolkit.analysis.plotting import SystemPlotter
from sd_toolkit.core.units import Q_, unit_registry

print("✅ System Dynamics Toolkit loaded successfully!")
print("🏢 Ready for economic indicators module testing")
print(f"📊 Available rate functions: {len(rate_registry.list_functions())}")

In [ ]:
# Load model structure and scenario parameters
print("📂 Loading economic indicators configuration...")

# Load model structure
model_structure_path = Path('model_structure.yaml')
with open(model_structure_path, 'r') as file:
    model_structure = yaml.safe_load(file)

print(f"✅ Model structure loaded: {model_structure['model']['name']}")

# Load scenario parameters
scenario_path = Path('scenario_parameters.yaml')
with open(scenario_path, 'r') as file:
    scenario_params = yaml.safe_load(file)

print(f"✅ Scenario parameters loaded: {scenario_params['scenario']['name']}")

# Display model dimensions
print("\n🏗️ Model Dimensions:")
for dim_name, dim_info in model_structure['dimensions'].items():
    print(f"  • {dim_name}: {dim_info['labels']} (size: {dim_info['size']})")

# Display external dependencies
print("\n🔗 External Dependencies:")
if 'external_dependencies' in model_structure:
    for dep in model_structure['external_dependencies']:
        print(f"  • {dep['name']}: from {dep['source_module']}.{dep['source_element']}")
else:
    print("  • No external dependencies defined")

In [ ]:
# Analyze initial economic conditions
print("📊 Analyzing initial economic conditions...")

# Extract initial employment data
employed_pop = np.array(scenario_params['constants']['employed_population_distribution'])
avg_income = np.array(scenario_params['constants']['average_income_distribution'])
participation_rates = np.array(scenario_params['constants']['labor_participation_rate'])

age_groups = model_structure['dimensions']['age_group']['labels']
genders = model_structure['dimensions']['gender']['labels']
income_brackets = model_structure['dimensions']['income_bracket']['labels']

print(f"\n👥 Initial Employment by Age and Gender:")
employment_df = pd.DataFrame(employed_pop, index=age_groups, columns=genders)
print(employment_df)

print(f"\n💰 Initial Average Income by Age and Income Bracket:")
income_df = pd.DataFrame(avg_income, index=age_groups, columns=income_brackets)
print(income_df)

print(f"\n📈 Labor Force Participation Rates:")
participation_df = pd.DataFrame(participation_rates, index=age_groups, columns=genders)
print(participation_df)

# Calculate summary statistics
total_employed = employed_pop.sum()
avg_participation = participation_rates[1:4].mean()  # Working age groups
income_range = avg_income[avg_income > 0]
median_income = np.median(income_range)

print(f"\n📋 Summary Statistics:")
print(f"  • Total Employed Population: {total_employed:,} persons")
print(f"  • Average Participation Rate (Working Age): {avg_participation:.1%}")
print(f"  • Median Income Level: ${median_income:,.0f}/year")

In [ ]:
# Create economic indicators visualizations
print("📊 Creating economic indicators visualizations...")

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Economic Indicators Module - Initial Conditions Analysis', fontsize=16, fontweight='bold')

# Plot 1: Employment by age group
ax1 = axes[0, 0]
employment_df.plot(kind='bar', ax=ax1, color=['skyblue', 'lightcoral'])
ax1.set_title('Employment by Age Group and Gender')
ax1.set_xlabel('Age Group')
ax1.set_ylabel('Employed Persons')
ax1.legend(title='Gender')
ax1.tick_params(axis='x', rotation=45)

# Plot 2: Income distribution by age group
ax2 = axes[0, 1]
income_df.plot(kind='bar', ax=ax2, color=['lightgreen', 'gold', 'orange'])
ax2.set_title('Average Income by Age Group and Income Bracket')
ax2.set_xlabel('Age Group')
ax2.set_ylabel('Income (USD/year)')
ax2.legend(title='Income Bracket')
ax2.tick_params(axis='x', rotation=45)

# Plot 3: Labor force participation rates
ax3 = axes[1, 0]
participation_df.plot(kind='bar', ax=ax3, color=['steelblue', 'salmon'])
ax3.set_title('Labor Force Participation Rates')
ax3.set_xlabel('Age Group')
ax3.set_ylabel('Participation Rate')
ax3.legend(title='Gender')
ax3.tick_params(axis='x', rotation=45)
ax3.set_ylim(0, 1)

# Plot 4: Economic output potential (employment × income)
ax4 = axes[1, 1]
# Calculate economic output potential for visualization
output_potential = []
for i, age_group in enumerate(age_groups):
    if i > 0:  # Skip 0-14 age group
        emp_total = employed_pop[i].sum()
        avg_inc = avg_income[i].mean()
        output_potential.append(emp_total * avg_inc / 1e6)  # Convert to millions
    else:
        output_potential.append(0)

ax4.bar(age_groups, output_potential, color='purple', alpha=0.7)
ax4.set_title('Economic Output Potential by Age Group')
ax4.set_xlabel('Age Group')
ax4.set_ylabel('Output Potential (Million USD/year)')
ax4.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Economic indicators visualizations created")

In [ ]:
# Test model building (without population dependency for now)
print("🔧 Testing economic indicators model building...")

# Create a simplified version for testing (without external dependencies)
test_config = model_structure.copy()
test_config['constants'] = scenario_params['constants']

# Remove external dependencies for standalone testing
if 'external_dependencies' in test_config:
    print("⚠️ Removing external dependencies for standalone testing")
    del test_config['external_dependencies']
    
    # Remove connections that depend on external elements
    test_config['connections'] = [
        conn for conn in test_config['connections'] 
        if conn['from'] != 'total_population'
    ]

# Create YAML system builder
builder = YAMLSystemBuilder()

# Build the model
try:
    economic_model = builder.build_from_dict(test_config)
    print(f"✅ Model built successfully: {economic_model.name}")
    print(f"📊 Model elements: {len(economic_model.elements)}")
    print(f"📦 Stocks: {len(economic_model.stocks)}")
    print(f"🌊 Flows: {len(economic_model.flows)}")
    print(f"🧮 Calculators: {len(economic_model.calculators)}")
    print(f"🔧 Auxiliaries: {len(economic_model.auxiliaries)}")
    
except Exception as e:
    print(f"❌ Error building model: {e}")
    economic_model = None

# Model validation summary
print("\n📋 Economic Indicators Module Validation Summary")
print("=" * 55)

validation_results = {
    "Model Structure Loading": "✅ Success" if model_structure else "❌ Failed",
    "Scenario Parameters Loading": "✅ Success" if scenario_params else "❌ Failed",
    "Initial Data Analysis": "✅ Success",
    "Visualizations": "✅ Success",
    "Model Building (Standalone)": "✅ Success" if economic_model else "❌ Failed"
}

for test, status in validation_results.items():
    print(f"  {test}: {status}")

print("\n🎯 Next Steps:")
if economic_model:
    print("  • ✅ Economic indicators module structure is valid")
    print("  • 🔗 Ready for integration with population dynamics module")
    print("  • 🏠 Can proceed with housing market module development")
else:
    print("  • ❌ Fix model building issues before proceeding")
    print("  • 🔍 Check parameter consistency and function definitions")
    print("  • 🛠️ Validate YAML syntax and dependency resolution")

print("\n🏁 Economic Indicators Module Test Complete!")